In [1]:
from dotenv import load_dotenv

load_dotenv()

True

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🧠 默认情况下，memory 是关闭的 ❌

**一句话定位**：每次 `agent.invoke()` 都是一次全新的、无状态的调用——除非你显式传入历史消息，agent 不会记得上一轮说过什么。

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：下面会连续问两次（先自我介绍，再问「我最喜欢的颜色是什么」），第二次 `invoke` 完全不会带上第一次的对话内容，模型会一脸懵——这不是 bug，是 `create_agent` 默认无状态的正常行为。

</div>

In [ ]:
from langchain.agents import create_agent


agent = create_agent("gpt-5-nano")

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(
    content="Hello my name is Seán and my favourite colour is green"
)

response = agent.invoke({"messages": [question]})

In [5]:
from pprint import pprint

pprint(response["messages"])

[HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='db1e45d7-ace8-4758-8dcd-961410db33fc'),
 AIMessage(content='Hi Seán! Nice to meet you. Green is a great color—calm, refreshing, and full of life. What would you like to do today? I can chat about something you enjoy, help you practice English, brainstorm ideas for a project with green themes, or share interesting facts about nature and green things.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 586, 'prompt_tokens': 18, 'total_tokens': 604, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXeN3FXHMzKYea6jYhJ0GzRTy5pna', 'service_tier': 'default', 

In [6]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke({"messages": [question]})

pprint(response)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='9b2c7ba0-b8c8-458d-8c40-57cd1db59ae2'),
              AIMessage(content='I don’t know your favourite colour. Do you want me to guess, or figure it out with a quick few questions?\n\nMy guess (random/common): blue.\n\nIf you’d like to determine it, I can ask a couple of quick questions (cool vs warm colors, bright vs muted, etc.). You can also tell me your actual favourite and I’ll remember it for this chat.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 792, 'prompt_tokens': 11, 'total_tokens': 803, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXeaDhxIbEZOAR

## 用 LangChain 启用 memory ✅

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**关键概念**：给 `create_agent` 传一个 `checkpointer`（比如 `InMemorySaver()`），agent 就会在每次 `invoke` 后把完整状态（包括消息历史）存下来，按 `thread_id` 区分不同的会话。只要后续调用传同一个 `config = {"configurable": {"thread_id": "1"}}`，之前的对话历史就会自动被带上，不需要手动拼接消息列表。

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：memory 的本质就是「checkpointer 按 thread_id 存取 state」——`InMemorySaver` 是最简单的进程内实现，生产环境换成 Postgres/Redis 等持久化 checkpointer 即可，接口不变。

</div>

In [7]:
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    "gpt-5-nano",
    checkpointer=InMemorySaver(),
)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(
    content="Hello my name is Seán and my favourite colour is green"
)
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,
)

In [9]:
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='8ccfd2f0-a631-45e5-9b6f-4e18b8a29562'),
              AIMessage(content='Hi Seán! Nice to meet you. Green is a great colour—very calming and full of life. What would you like to do today? I can chat about green things, help with Gaelic, tell a story, or answer any questions you have.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 509, 'prompt_tokens': 18, 'total_tokens': 527, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXebN9H1akh72GZjzDD5J1i5uTGls', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run

In [10]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='8ccfd2f0-a631-45e5-9b6f-4e18b8a29562'),
              AIMessage(content='Hi Seán! Nice to meet you. Green is a great colour—very calming and full of life. What would you like to do today? I can chat about green things, help with Gaelic, tell a story, or answer any questions you have.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 509, 'prompt_tokens': 18, 'total_tokens': 527, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXebN9H1akh72GZjzDD5J1i5uTGls', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run